# 03 — VL-JEPA Pretraining

Self-Supervised Representation Learning on 3D CT patches.

Pipeline:
context → online encoder → predictor → z_pred
target → momentum encoder → z_target (stop-grad)

Loss:
SmoothL1(z_pred, z_target)

EMA update for momentum encoder.

No labels used.


In [1]:
import sys
from pathlib import Path

# Add project root to Python path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

# If running from notebooks folder, use this instead:
# PROJECT_ROOT = Path().resolve().parents[1]
# sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)


Project root added: /home/sreethanu/Downloads/lung_cancer_vljepa


In [10]:
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from torch.utils.data import DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from tqdm import tqdm

from src.models.encoder_3d import ViT3DEncoder
from src.models.momentum_encoder import MomentumEncoder, get_momentum_schedule
from src.models.jepa_predictor import JEPAPredictor
from src.models.masking import BlockMask3D, ContextMask3D
from src.utils.losses import JEPALoss
from src.utils.config import Config
from src.utils.seed import set_global_seed
from src.utils.ssl_dataset import SSLDataset


In [3]:
config_path = PROJECT_ROOT / "configs" / "jepa_config.yaml"
config = Config(str(config_path))

set_global_seed(
    config.get("project.seed"),
    deterministic=config.get("project.deterministic")
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Setting global seed: 42
Enabling deterministic mode (may reduce performance)
Reproducibility configured successfully.
Device: cuda


In [4]:
patch_root = PROJECT_ROOT / "data" / "processed" / "patches"
patch_paths = sorted(patch_root.rglob("*.npz"))

print("Total patches found:", len(patch_paths))

if len(patch_paths) == 0:
    raise RuntimeError(f"No .npz patches found under {patch_root}")

seed_value = config.get("project.seed", 42)
generator = torch.Generator().manual_seed(seed_value)
indices = torch.randperm(len(patch_paths), generator=generator).tolist()

val_size = max(1, int(0.1 * len(indices)))
train_size = len(indices) - val_size

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(SSLDataset(patch_paths, augment=True), train_indices)
val_dataset = Subset(SSLDataset(patch_paths, augment=False), val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=True,
    num_workers=config.get("data.num_workers"),
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.get("data.batch_size"),
    shuffle=False,
    num_workers=config.get("data.num_workers"),
    pin_memory=True,
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Total patches found: 12750
Train samples: 11475
Val samples: 1275
Train batches: 1913
Val batches: 213


In [5]:
encoder_embed_dim = min(config.get("model.encoder.embed_dim", 384), 384)
encoder_depth = min(config.get("model.encoder.depth", 6), 6)
encoder_num_heads = min(config.get("model.encoder.num_heads", 6), 6)

predictor_embed_dim = min(config.get("model.predictor.embed_dim", 192), 192)
predictor_depth = min(config.get("model.predictor.depth", 4), 4)
predictor_num_heads = min(config.get("model.predictor.num_heads", 4), 4)

online_encoder = ViT3DEncoder(
    input_size=tuple(config.get("data.input_shape", [1, 96, 96, 96])[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size", [16, 16, 16])),
    embed_dim=encoder_embed_dim,
    depth=encoder_depth,
    num_heads=encoder_num_heads,
).to(device)

target_encoder = MomentumEncoder(
    base_encoder=online_encoder,
    momentum=config.get("model.jepa.ema_decay_start")
).to(device)

predictor = JEPAPredictor(
    embed_dim=encoder_embed_dim,
    predictor_embed_dim=predictor_embed_dim,
    depth=predictor_depth,
    num_heads=predictor_num_heads,
    max_num_patches=online_encoder.get_num_patches(),
).to(device)

print("Online encoder params:", sum(p.numel() for p in online_encoder.parameters()))
print("Predictor params:", sum(p.numel() for p in predictor.parameters()))


Online encoder params: 12303744
Predictor params: 1969536


In [6]:
target_masker = BlockMask3D(
    input_size=tuple(config.get("data.input_shape", [1, 96, 96, 96])[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size", [16, 16, 16])),
    num_blocks=config.get("model.masking.num_blocks", 4),
    min_block_scale=max(config.get("model.masking.min_block_scale", 0.30), 0.30),
    max_block_scale=max(config.get("model.masking.max_block_scale", 0.50), 0.50),
    aspect_ratio_min=0.75,
    
    aspect_ratio_max=1.50,
)

context_masker = ContextMask3D(
    input_size=tuple(config.get("data.input_shape", [1, 96, 96, 96])[1:]),
    patch_size=tuple(config.get("model.encoder.patch_size", [16, 16, 16])),
    keep_ratio_min=config.get("model.masking.context_keep_ratio_min", 0.40),
    keep_ratio_max=config.get("model.masking.context_keep_ratio_max", 0.60),
)

criterion = JEPALoss(loss_type="smooth_l1")


In [7]:
params = list(online_encoder.parameters()) + list(predictor.parameters())

optimizer = torch.optim.AdamW(
    params,
    lr=config.get("pretraining.learning_rate"),
    weight_decay=config.get("pretraining.weight_decay"),
)

warmup_epochs = config.get("model.jepa.ema_warmup_epochs", 10)  # reuse the same warmup value

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1e-2,
    end_factor=1.0,
    total_iters=warmup_epochs,
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.get("pretraining.epochs") - warmup_epochs,
    eta_min=config.get("pretraining.min_lr"),
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs],
)

momentum_schedule = get_momentum_schedule(
    base_momentum=config.get("model.jepa.ema_decay_start"),
    final_momentum=config.get("model.jepa.ema_decay_end"),
    epochs=config.get("pretraining.epochs"),
    warmup_epochs=config.get("model.jepa.ema_warmup_epochs"),
)

scaler = GradScaler(enabled=config.get("pretraining.amp"))

In [8]:
volumes = next(iter(train_loader))
volumes = volumes.to(device)
B = volumes.size(0)

_, target_mask = target_masker(B)
visible_mask = context_masker(B, target_masks=target_mask)

target_mask = target_mask.to(device)
visible_mask = visible_mask.to(device)

with torch.no_grad():
    context_latents = online_encoder(volumes, mask=visible_mask)
    target_latents = target_encoder(volumes)
    predicted_latents = predictor(
        context_latents,
        target_mask,
        online_encoder.get_num_patches(),
        visible_mask=visible_mask
    )

example_targets = target_latents[0][target_mask[0]]
example_preds = predicted_latents[0][:example_targets.size(0)]

print("Context latents shape:", tuple(context_latents.shape))
print("Predicted latents shape:", tuple(predicted_latents.shape))
print("Target latents shape:", tuple(target_latents.shape))
print("Example masked targets shape:", tuple(example_targets.shape))
print("Example masked preds shape:", tuple(example_preds.shape))
print("Mask ratio (batch mean):", target_mask.float().mean().item())
print("Visible ratio (batch mean):", visible_mask.float().mean().item())


Context latents shape: (6, 71, 384)
Predicted latents shape: (6, 95, 384)
Target latents shape: (6, 216, 384)
Example masked targets shape: (81, 384)
Example masked preds shape: (81, 384)
Mask ratio (batch mean): 0.38040122389793396
Visible ratio (batch mean): 0.264660507440567


In [11]:
save_dir = PROJECT_ROOT / "checkpoints" / "pretraining"
save_dir.mkdir(parents=True, exist_ok=True)

epochs = config.get("pretraining.epochs")
save_freq = config.get("pretraining.save_freq")

best_val_loss = float("inf")
train_loss_history = []
val_loss_history = []

print("Starting JEPA pretraining...")


def compute_masked_loss(z_pred, z_target, target_mask):
    masked_targets = []
    masked_preds = []

    for b in range(target_mask.size(0)):
        tgt = z_target[b][target_mask[b]]
        pred = z_pred[b][:tgt.size(0)]

        if tgt.numel() == 0:
            continue

        masked_targets.append(tgt)
        masked_preds.append(pred)

    if not masked_targets:
        raise RuntimeError("No masked tokens found in current batch.")

    masked_targets = torch.cat(masked_targets, dim=0)
    masked_preds = torch.cat(masked_preds, dim=0)

    # ↓ REPLACE these two original lines:
    # loss_value = criterion(masked_preds, masked_targets)
    # return loss_value, masked_preds, masked_targets

    # ↓ WITH these three lines:
    masked_targets = F.layer_norm(masked_targets, [masked_targets.shape[-1]])
    loss_value = criterion(masked_preds, masked_targets.detach())
    return loss_value, masked_preds, masked_targets


for epoch in range(epochs):

    online_encoder.train()
    predictor.train()

    total_train_loss = 0.0

    for volumes in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - train"):
        volumes = volumes.to(device, non_blocking=True)
        batch_size = volumes.size(0)

        _, target_mask_cpu = target_masker(batch_size)
        visible_mask_cpu = context_masker(batch_size, target_masks=target_mask_cpu)

        target_mask = target_mask_cpu.to(device)
        visible_mask = visible_mask_cpu.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=config.get("pretraining.amp")):
            context_latents = online_encoder(volumes, mask=visible_mask)

            with torch.no_grad():
                target_latents = target_encoder(volumes)

            predicted_latents = predictor(
                context_latents,
                target_mask,
                online_encoder.get_num_patches(),
                visible_mask=visible_mask,
            )

            loss, _, _ = compute_masked_loss(predicted_latents, target_latents, target_mask)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer) 
        torch.nn.utils.clip_grad_norm_(
            params,
            config.get("pretraining.gradient_clip_val"),
        )

        scaler.step(optimizer)
        scaler.update()

        # EMA update (target encoder never receives gradients)
        momentum = momentum_schedule[epoch]
        target_encoder.update(online_encoder, momentum)

        total_train_loss += loss.item()

    scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)
    train_loss_history.append(avg_train_loss)

    # Validation (masked-token latent loss only)
    online_encoder.eval()
    predictor.eval()

    total_val_loss = 0.0

    with torch.no_grad():
        for volumes in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - val", leave=False):
            volumes = volumes.to(device, non_blocking=True)
            batch_size = volumes.size(0)

            _, target_mask_cpu = target_masker(batch_size)
            visible_mask_cpu = context_masker(batch_size, target_masks=target_mask_cpu)

            target_mask = target_mask_cpu.to(device)
            visible_mask = visible_mask_cpu.to(device)

            with autocast(enabled=config.get("pretraining.amp")):
                context_latents = online_encoder(volumes, mask=visible_mask)
                target_latents = target_encoder(volumes)
                predicted_latents = predictor(
                    context_latents,
                    target_mask,
                    online_encoder.get_num_patches(),
                    visible_mask=visible_mask,
                )

                val_loss, _, _ = compute_masked_loss(predicted_latents, target_latents, target_mask)

            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_loss_history.append(avg_val_loss)

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Val Loss: {avg_val_loss:.6f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(
            {
                "epoch": epoch,
                "best_val_loss": best_val_loss,
                "online_encoder": online_encoder.state_dict(),
                "target_encoder": target_encoder.encoder.state_dict(),
                "predictor": predictor.state_dict(),
                "optimizer": optimizer.state_dict(),
            },
            save_dir / "best_model.pth",
        )
        print("  -> Best model saved.")

    if (epoch + 1) % save_freq == 0:
        
        torch.save(
            {
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "online_encoder": online_encoder.state_dict(),
                "target_encoder": target_encoder.encoder.state_dict(),
                "predictor": predictor.state_dict(),
                "optimizer": optimizer.state_dict(),
            },
            save_dir / f"jepa_epoch_{epoch+1}.pth",
        )

        print("Checkpoint saved.")

print("\nFinal train loss:", train_loss_history[-1])
print("Final val loss:", val_loss_history[-1])
print("Best val loss:", best_val_loss)


Starting JEPA pretraining...


Epoch 1/100 - train: 100%|██████████████████| 1913/1913 [01:24<00:00, 22.61it/s]
                                                                                

Epoch [1/100] | Train Loss: 0.272826 | Val Loss: 0.204340
  -> Best model saved.


Epoch 2/100 - train: 100%|██████████████████| 1913/1913 [01:25<00:00, 22.31it/s]
                                                                                

Epoch [2/100] | Train Loss: 0.118025 | Val Loss: 0.169659
  -> Best model saved.


...

Epoch [98/100] | Train Loss: 0.036227 | Val Loss: 0.038305


Epoch 99/100 - train: 100%|█████████████████| 1913/1913 [01:25<00:00, 22.44it/s]
                                                                                

Epoch [99/100] | Train Loss: 0.036175 | Val Loss: 0.038171
  -> Best model saved.


Epoch 100/100 - train: 100%|████████████████| 1913/1913 [01:25<00:00, 22.36it/s]
                                                                                

Epoch [100/100] | Train Loss: 0.036132 | Val Loss: 0.038411
Checkpoint saved.

Final train loss: 0.03613241193003045
Final val loss: 0.03841054795458563
Best val loss: 0.03817140967738181


In [12]:
from pathlib import Path
import os

save_dir = PROJECT_ROOT / "checkpoints" / "pretraining"

print("Checking directory:", save_dir)

if save_dir.exists():
    print(" Pretraining directory exists.")
    
    checkpoint_files = list(save_dir.glob("*.pth"))
    
    if len(checkpoint_files) == 0:
        print(" No checkpoint files found.")
    else:
        print(f"✔ Found {len(checkpoint_files)} checkpoint file(s):")
        for file in checkpoint_files:
            size_mb = os.path.getsize(file) / (1024 * 1024)
            print(f"   - {file.name} | {size_mb:.2f} MB")

else:
    print(" Pretraining directory does NOT exist.")


Checking directory: /home/sreethanu/Downloads/lung_cancer_vljepa/checkpoints/pretraining
 Pretraining directory exists.
✔ Found 11 checkpoint file(s):
   - jepa_epoch_10.pth | 210.46 MB
   - jepa_epoch_70.pth | 210.46 MB
   - jepa_epoch_60.pth | 210.46 MB
   - jepa_epoch_90.pth | 210.46 MB
   - jepa_epoch_30.pth | 210.46 MB
   - jepa_epoch_20.pth | 210.46 MB
   - jepa_epoch_80.pth | 210.46 MB
   - jepa_epoch_50.pth | 210.46 MB
   - best_model.pth | 210.46 MB
   - jepa_epoch_40.pth | 210.46 MB
   - jepa_epoch_100.pth | 210.46 MB


In [13]:
best_model_path = save_dir / "best_model.pth"

if best_model_path.exists():
    size_mb = os.path.getsize(best_model_path) / (1024 * 1024)
    print(f"✔ best_model.pth exists | {size_mb:.2f} MB")
else:
    print("⚠ best_model.pth not found.")


✔ best_model.pth exists | 210.46 MB


In [15]:
checkpoints_dir = PROJECT_ROOT / "checkpoints" / "pretraining"
if checkpoints_dir.exists():
    print("✔ Checkpoints directory exists.")
    print("Files inside:", [f.name for f in checkpoints_dir.iterdir()])
else:
    print("✘ No checkpoints found — something went wrong.")


✔ Checkpoints directory exists.
Files inside: ['jepa_epoch_10.pth', 'jepa_epoch_70.pth', 'jepa_epoch_60.pth', 'jepa_epoch_90.pth', 'jepa_epoch_30.pth', 'jepa_epoch_20.pth', 'jepa_epoch_80.pth', 'jepa_epoch_50.pth', 'best_model.pth', 'jepa_epoch_40.pth', 'jepa_epoch_100.pth']


In [16]:
best_path = PROJECT_ROOT / "checkpoints" / "pretraining" / "best_model.pth"

if best_path.exists():
    checkpoint_path = best_path
else:
    checkpoint_path = PROJECT_ROOT / "checkpoints" / "pretraining" / "jepa_epoch_100.pth"

checkpoint = torch.load(checkpoint_path, map_location=device)

online_encoder.load_state_dict(checkpoint["online_encoder"])
online_encoder.eval()

print(f"Pretrained encoder loaded successfully from: {checkpoint_path.name}")


Pretrained encoder loaded successfully from: best_model.pth
